# Fine-tuning a BERT-style model for Question Answering (free GPU)

This notebook fine-tunes **DistilBERT** on the **SQuAD v2.0** dataset (130k questions, one third unanswerable) and evaluates it with **Exact Match (accuracy)** and **F1-score**.

**Before you start:** `Runtime` → `Change runtime type` → select **T4 GPU** → `Save`.

Total time: about 1 hour on the free T4 GPU.

## 1. Get the project code
Run the cell below and click **Choose Files**, then pick `project_for_colab.zip` from your project folder.

(If your project is already on GitHub, you can instead use the commented `git clone` line.)

In [ ]:
from google.colab import files
files.upload()  # choose project_for_colab.zip
!unzip -q -o project_for_colab.zip -d Question-Answering-System

# Alternative if the project is on GitHub:
# !git clone https://github.com/pujithavarma17/Question-Answering-System.git

%cd Question-Answering-System

## 2. Install libraries and check the GPU
Colab already has PyTorch with GPU support, so we only add what is missing.
`transformers` is pinned to 5.2.0 because newer versions removed the question-answering pipeline.

In [ ]:
!pip install -q "transformers==5.2.0" datasets accelerate sentence-transformers pypdf
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3. Fine-tune
Starts from the general pre-trained `distilbert-base-uncased` model and trains it on all of SQuAD v2.0 for 2 epochs.

The model learns to predict the **start** and **end** token of the answer; for unanswerable questions it predicts the `[CLS]` token.

In [ ]:
!python scripts/train_qa.py --base_model distilbert-base-uncased --train_samples 0 --epochs 2 --output_dir models/qa-distilbert-squad2

## 4. Evaluate the fine-tuned model
Three test sets:
* **squad** – 500 questions from the official SQuAD v2.0 validation set (never seen in training)
* **custom** – our 60 hand-written questions (education, customer support, operating systems)
* **document** – 20 questions about a full document, answered with the complete app pipeline

In [ ]:
for data in ["squad", "custom", "document"]:
    !python scripts/evaluate_qa.py --model models/qa-distilbert-squad2 --data {data} --n 500

## 5. (Optional) Experiment: a bigger model
BERT-base has 110M parameters vs 66M for DistilBERT. Expect higher accuracy but about 2x slower training (~2 hours).

In [ ]:
# Uncomment to run
# !python scripts/train_qa.py --base_model bert-base-uncased --train_samples 0 --epochs 2 --output_dir models/qa-bert-base-squad2
# for data in ["squad", "custom", "document"]:
#     !python scripts/evaluate_qa.py --model models/qa-bert-base-squad2 --data {data} --n 500

## 6. Results table

In [ ]:
!python scripts/make_results_table.py

## 7. Save your model (free)

**Option A – Hugging Face Hub (recommended, lets the deployed app use your model).**
1. Create a free account at https://huggingface.co
2. Go to *Settings → Access Tokens → Create new token* with **Write** permission and copy it.
3. Run the next cell and paste the token when asked.

In [ ]:
from huggingface_hub import HfApi, login

login()  # paste your token

HF_USERNAME = "YOUR-HF-USERNAME"
repo_id = f"{HF_USERNAME}/qa-distilbert-squad2"

api = HfApi()
api.create_repo(repo_id, exist_ok=True)
api.upload_folder(folder_path="models/qa-distilbert-squad2", repo_id=repo_id)
print("Model uploaded: https://huggingface.co/" + repo_id)

**Option B – Download to your laptop.** Unzip it inside your project folder (it contains `models/` and `results/`). Then the evaluation scripts can use `--model models/qa-distilbert-squad2`.

In [ ]:
!zip -qr qa-model-and-results.zip models/qa-distilbert-squad2 results
from google.colab import files
files.download("qa-model-and-results.zip")